In [1]:
import torch
import os
from torchvision import models
import torch.nn as nn
from pathlib import Path

# ── Load lại MobileNetV2 từ checkpoint ───────────────────────────────────
DEVICE   = torch.device('cpu')  # Dùng CPU để export, không cần GPU
OUT_DIR  = Path('/kaggle/working')
CKPT     = '/kaggle/input/notebooks/tk1774/train-vslv2/mobilenetv2_checkpoint.pth'

# Rebuild model
def build_mobilenetv2(num_classes=25, dropout=0.3):
    from torchvision.models import MobileNet_V2_Weights
    model = models.mobilenet_v2(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=dropout / 2),
        nn.Linear(512, 25)
    )
    return model

ckpt  = torch.load(CKPT, map_location=DEVICE)
model = build_mobilenetv2()
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print('✅ Load checkpoint thành công')
print(f'Classes: {ckpt["classes"]}')

✅ Load checkpoint thành công
Classes: ['A', 'B', 'C', 'D', 'E', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'Râu', 'S', 'T', 'U', 'V', 'X', 'Y', 'mũ', 'Đ']


In [2]:
# ── Export ONNX (single file) ─────────────────────────────────────────────
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model, dummy,
    str(OUT_DIR / 'mobilenetv2_vsl_v2.onnx'),
    opset_version=17, dynamo=False,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}}
)
size = os.path.getsize(OUT_DIR / 'mobilenetv2_vsl_v2.onnx') / 1e6
print(f'✅ ONNX: {size:.1f} MB')

/tmp/ipykernel_57/3356700556.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


✅ ONNX: 11.5 MB


In [4]:
# ── Dùng onnx2tf thay thế onnx-tf ────────────────────────────────────────
!pip install onnx2tf tensorflow -q

import onnx2tf
import tensorflow as tf
import os

# ONNX → TFLite trực tiếp (không cần qua SavedModel thủ công)
onnx2tf.convert(
    input_onnx_file_path=str(OUT_DIR / 'mobilenetv2_vsl_v2.onnx'),
    output_folder_path=str(OUT_DIR / 'mbv2_tflite_out'),
    output_tfv1_pb=False,
    non_verbose=True,
)

# File .tflite nằm trong thư mục output
import glob
tflite_files = glob.glob(str(OUT_DIR / 'mbv2_tflite_out/*.tflite'))
print(f'TFLite files: {tflite_files}')

for f in tflite_files:
    size = os.path.getsize(f) / 1e6
    print(f'  {os.path.basename(f)}: {size:.1f} MB')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 1.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 12.1 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 19.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 20.0 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 55.9 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 40.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 45.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 70.3 MB

flatbuffer_direct lowering:   0%|          | 0/102 [00:00<?, ?it/s]

flatbuffer_direct post-lowering:   0%|          | 0/6 [00:00<?, ?it/s]

flatbuffer_direct export:   0%|          | 0/3 [00:00<?, ?it/s]

flatbuffer_direct write timing: stage=float32 mode=builder_direct total=0.063s serialize=0.056s (sanitize=0.000s build=0.010s pack=0.044s output=0.002s) write=0.007s size=11.00MB
flatbuffer_direct write timing: stage=float16 mode=builder_direct total=0.037s serialize=0.033s (sanitize=0.000s build=0.006s pack=0.026s output=0.001s) write=0.003s size=5.51MB
TFLite files: ['/kaggle/working/mbv2_tflite_out/mobilenetv2_vsl_v2_float32.tflite', '/kaggle/working/mbv2_tflite_out/mobilenetv2_vsl_v2_float16.tflite']
  mobilenetv2_vsl_v2_float32.tflite: 11.5 MB
  mobilenetv2_vsl_v2_float16.tflite: 5.8 MB
